<a href="https://colab.research.google.com/github/BF667-IDLE/vsep/blob/main/notebooks/vsep_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vsep — Audio Stem Separator

Separate any audio into stems (vocals, drums, bass, etc.) using state-of-the-art AI models from UVR.
**83 models** across Roformer, MDX-Net, MDXC, VR, and Demucs architectures.

---

In [ ]:
#@title 1. Install
#@markdown Clone repo, install deps, yt-dlp, ffmpeg

!git clone -q https://github.com/BF667-IDLE/vsep.git /content/vsep
!pip install -q -r /content/vsep/requirements.txt yt-dlp
!apt-get -qq install ffmpeg > /dev/null 2>&1

import os; os.chdir("/content/vsep")
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "No GPU — go to Runtime > Change runtime type")
import yt_dlp; print(f"yt-dlp {yt_dlp.version.__version__} ready")

In [ ]:
#@title 2. Get Audio
#@markdown Upload a file OR paste a YouTube/SoundCloud URL

import os, glob
audio_source = "Upload file" #@param ["Upload file", "YouTube / URL"]

if audio_source == "Upload file":
    from google.colab import files
    print("Upload your audio (MP3, WAV, FLAC, OGG, M4A)...")
    uploaded = files.upload()
    if uploaded:
        audio_file = list(uploaded.keys())[0]
    else:
        raise SystemExit("No file uploaded.")
else:
    url = "" #@param {type:"string"}
    if not url.strip():
        raise SystemExit("Paste a URL first.")
    os.makedirs("/content/vsep/audio_input", exist_ok=True)
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": "/content/vsep/audio_input/%(title).80s.%(ext)s",
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav", "preferredquality": "0"}],
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url.strip()])
    audio_file = sorted(glob.glob("/content/vsep/audio_input/*.wav"))[-1]

print(f"\nAudio: {audio_file} ({os.path.getsize(audio_file)/1e6:.1f} MB)")

In [ ]:
#@title 3. Select Model
#@markdown 83 models from UVR. Display name maps to actual filename.

MODEL_CATALOG = {
    # --- Vocals ---
    "BS Roformer EP 317 SDR 12.98":      "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
    "BS Roformer EP 368 SDR 12.96":      "model_bs_roformer_ep_368_sdr_12.9628.ckpt",
    "Mel Roformer Kimberly Jensen":       "vocals_mel_band_roformer.ckpt",
    "Mel Roformer Viperx SDR 12.61":     "Mel-Roformer-Viperx-1053.ckpt",
    "BS Roformer Viperx 1297":           "BS-Roformer-Viperx-1297.ckpt",
    "BS Roformer Vocals Gabox":          "bs_roformer_vocals_gabox.ckpt",
    "MelBand Roformer Vocals Gabox":     "mel_band_roformer_vocals_gabox.ckpt",
    "MelBand Roformer Vocals V2 Gabox":  "mel_band_roformer_vocals_v2_gabox.ckpt",
    "MelBand Roformer Vocals FV3 Gabox": "mel_band_roformer_vocals_fv3_gabox.ckpt",
    "MelBand Roformer Vocals FV6 Gabox": "mel_band_roformer_vocals_fv6_gabox.ckpt",
    "BS Roformer Vocals Revive V3":      "bs_roformer_vocals_revive_v3e_unwa.ckpt",
    "BS Roformer Vocals Resurrection":   "bs_roformer_vocals_resurrection_unwa.ckpt",
    "MelBand Roformer SYHFT V3":         "MelBandRoformerSYHFTV3Epsilon.ckpt",
    "MelBand Roformer Big Beta 6X":      "melband_roformer_big_beta6x.ckpt",
    "MDX23C InstVoc HQ SDR 11.95":      "MDX23C-8KFFT-InstVoc_HQ.ckpt",
    "MDX23C InstVoc SDR 10.75":         "MDX23C-8KFFT-InstVoc.ckpt",
    "MDX Net Inst HQ 2 SDR 10.93":      "UVR-MDX-NET-Inst_HQ_2.onnx",
    "MDX Net Inst 1 SDR 10.65":         "UVR-MDX-NET-Inst_1.onnx",
    "MDX Net Inst HQ 4 SDR 10.49":      "UVR-MDX-NET-Inst_HQ_4.onnx",
    "MDX Net Inst HQ 5":                "UVR-MDX-NET-Inst_HQ_5.onnx",
    "Kim Vocal 2":                      "Kim_Vocal_2.onnx",
    "MDX Net Karaoke 2":                "UVR-MDX-NET-KARA_2.onnx",
    "VR 5 HP":                          "5_HP-UVR.pth",

    # --- Instrumental / Karaoke ---
    "MelBand Instrumental becruily":    "mel_band_roformer_instrumental_becruily.ckpt",
    "MelBand Instrumental Gabox":       "mel_band_roformer_instrumental_gabox.ckpt",
    "MelBand Instrumental 2 Gabox":     "mel_band_roformer_instrumental_2_gabox.ckpt",
    "MelBand Instrumental 3 Gabox":     "mel_band_roformer_instrumental_3_gabox.ckpt",
    "MelBand Instrumental FV8 Gabox":   "mel_band_roformer_instrumental_fv8_gabox.ckpt",
    "MelBand Instrumental FVX Gabox":   "mel_band_roformer_instrumental_fvx_gabox.ckpt",
    "MelBand INST V8 Gabox":            "MelBandRoformerINSTV8.ckpt",
    "MelBand Inst V1 Plus Unwa":        "melband_roformer_inst_v1_plus.ckpt",
    "MelBand Inst V1E Plus Unwa":       "melband_roformer_inst_v1e_plus.ckpt",
    "BS Roformer Instrumental Resurrect":"bs_roformer_instrumental_resurrection_unwa.ckpt",
    "MelRoformer Karaoke Aufr33":        "mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt",
    "MelBand Karaoke Gabox":            "mel_band_roformer_karaoke_gabox.ckpt",
    "MelBand Karaoke V2 Gabox":         "mel_band_roformer_karaoke_v2_gabox.ckpt",
    "MelBand Karaoke becruily":         "mel_band_roformer_karaoke_becruily.ckpt",

    # --- Multi-Stem (Demucs) ---
    "Demucs v4 Fine Tuned":             "ht-demucs_ft.yaml",
    "Demucs v4 htdemucs":              "htdemucs.yaml",
    "Demucs v4 hdemucs mmi":           "hdemucs_mmi.yaml",
    "Demucs v4 6 Stem":                "htdemucs_6s.yaml",

    # --- Reverb Removal ---
    "Reverb HQ SDR 11.20 (MDXC)":      "MDX23C-8KFFT-Reverb_HQ.ckpt",
    "Echo HQ SDR 11.10 (MDXC)":        "MDX23C-8KFFT-Echo_HQ.ckpt",
    "MDX23C De-Reverb aufr33":         "MDX23C-De-Reverb-aufr33-jarredou.ckpt",
    "BS Roformer De-Reverb":            "deverb_bs_roformer_8_384dim_10depth.ckpt",
    "MelBand De-Reverb anvuew":        "dereverb_mel_band_roformer_anvuew_sdr_19.1729.ckpt",
    "MelBand De-Reverb Less Aggressive":"dereverb_mel_band_roformer_less_aggressive_anvuew_sdr_18.8050.ckpt",
    "MelBand De-Reverb Mono anvuew":   "dereverb_mel_band_roformer_mono_anvuew.ckpt",
    "MelBand De-Reverb Big Sucial":    "dereverb_big_mbr_ep_362.ckpt",
    "MelBand De-Reverb Super Big":      "dereverb_super_big_mbr_ep_346.ckpt",
    "MelBand De-Reverb Echo Sucial":   "melband_roformer_dereverb_echo.ckpt",
    "MelBand De-Reverb Echo V2":       "melband_roformer_dereverb_echo_v2.ckpt",
    "MelBand De-Reverb Echo Fused":    "melband_roformer_dereverb_echo_fused.ckpt",

    # --- De-Esser ---
    "De-Esser MDX SDR 10.90":           "MDX23C-8KFFT-DeEsser.ckpt",

    # --- Denoise / Debleed ---
    "MelRoformer Denoise aufr33":       "denoise_mel_band_roformer_aufr33_sdr_27.9959.ckpt",
    "MelRoformer Denoise Aggressive":   "denoise_mel_band_roformer_aufr33_aggr_sdr_27.9768.ckpt",
    "MelBand Denoise Debleed Gabox":    "mel_band_roformer_denoise_debleed_gabox.ckpt",
    "MelBand Bleed Suppressor V1":     "mel_band_roformer_bleed_suppressor_v1.ckpt",
    "MelBand Instrumental Bleedless V3":"mel_band_roformer_instrumental_bleedless_v3_gabox.ckpt",

    # --- Drum / Bass / Guitar / Piano ---
    "MDXC Guitar SDR 10.80":            "MDX23C-8KFFT-Guitar.ckpt",
    "MDXC Drum aufr33":                "MDX23C-DrumSep-aufr33-jarredou.ckpt",
    "MDXC Bass SDR 10.10":             "MDX23C-8KFFT-Bass.ckpt",
    "MDXC Piano SDR 9.90":             "MDX23C-8KFFT-Piano.ckpt",

    # --- Special ---
    "MelRoformer Crowd Removal":        "mel_band_roformer_crowd_aufr33_viperx_sdr_8.7144.ckpt",
    "BS Roformer Male-Female":          "bs_roformer_male_female_by_aufr33_sdr_7.2889.ckpt",
    "BS Roformer Chorus Male-Female":   "model_chorus_bs_roformer_ep_267_sdr_24.1275.ckpt",
    "MelBand Aspiration Sucial":        "aspiration_mel_band_roformer_sdr_18.9845.ckpt",
    "MelBand Aspiration Less Aggressive":"aspiration_mel_band_roformer_less_aggr_sdr_18.1201.ckpt",
    "VR De-Reverb aufr33":              "UVR-De-Reverb-aufr33-jarredou.pth",
    "VR BVE 4B SN":                    "UVR-BVE-4B_SN-44100-2.pth",
    "BS Roformer SW jarredou":          "bs_roformer_sw.ckpt",
}

model = "BS Roformer EP 317 SDR 12.98" #@param [
    "BS Roformer EP 317 SDR 12.98", "BS Roformer EP 368 SDR 12.96",
    "Mel Roformer Kimberly Jensen", "Mel Roformer Viperx SDR 12.61", "BS Roformer Viperx 1297",
    "BS Roformer Vocals Gabox", "MelBand Roformer Vocals Gabox", "MelBand Roformer Vocals V2 Gabox",
    "BS Roformer Vocals Revive V3", "BS Roformer Vocals Resurrection",
    "MelBand Roformer SYHFT V3", "MelBand Roformer Big Beta 6X",
    "MDX23C InstVoc HQ SDR 11.95", "MDX23C InstVoc SDR 10.75",
    "MDX Net Inst HQ 2 SDR 10.93", "MDX Net Inst HQ 5",
    "MelBand Instrumental becruily", "MelBand Instrumental Gabox", "MelBand INST V8 Gabox",
    "MelBand Inst V1 Plus Unwa", "BS Roformer Instrumental Resurrect",
    "MelRoformer Karaoke Aufr33", "MelBand Karaoke Gabox", "MelBand Karaoke V2 Gabox",
    "Demucs v4 Fine Tuned", "Demucs v4 htdemucs", "Demucs v4 6 Stem",
    "Reverb HQ SDR 11.20 (MDXC)", "Echo HQ SDR 11.10 (MDXC)",
    "BS Roformer De-Reverb", "MelBand De-Reverb anvuew", "MelBand De-Reverb Big Sucial",
    "MelBand De-Reverb Echo Sucial",
    "De-Esser MDX SDR 10.90",
    "MelRoformer Denoise aufr33", "MelBand Denoise Debleed Gabox",
    "MDXC Guitar SDR 10.80", "MDXC Drum aufr33", "MDXC Bass SDR 10.10", "MDXC Piano SDR 9.90",
    "MelRoformer Crowd Removal", "BS Roformer Male-Female", "BS Roformer Chorus Male-Female",
    "MelBand Aspiration Sucial", "VR De-Reverb aufr33", "BS Roformer SW jarredou",
]
selected_model = MODEL_CATALOG[model]
print(f"Model: {model}")
print(f"File:  {selected_model}")
print(f"Total models available: {len(MODEL_CATALOG)}")

In [ ]:
#@title 4. Separate
#@markdown Run the AI separation on your audio

from separator import Separator
import config.variables as cfg

cfg.MAX_DOWNLOAD_WORKERS = 4
cfg.DOWNLOAD_CHUNK_SIZE = 262144

print(f"Separating {audio_file} with {selected_model}...")
print("(downloading model on first use, cached after)")

separator = Separator(
    model_file_dir="/content/vsep/models",
    output_dir="/content/vsep/output",
    output_format="WAV",
    use_soundfile=True,
)
separator.load_model(model_filename=selected_model)
output_files = separator.separate(audio_file)

print(f"\nDone! {len(output_files)} stem(s):")
for f in output_files:
    print(f"  {f}")

In [ ]:
#@title 5. Listen & Download
#@markdown Play the results and download to your device

import glob
from IPython.display import Audio, display
from google.colab import files as dl

stems = sorted(glob.glob("/content/vsep/output/*.*"))
if not stems:
    raise SystemExit("No stems found. Run the separation cell first.")

for i, path in enumerate(stems, 1):
    name = os.path.basename(path)
    size = os.path.getsize(path) / 1e6
    print(f"{i}. {name} ({size:.1f} MB)")
    display(Audio(path))

print("\n--- Downloading all stems ---")
for path in stems:
    dl.download(path)